In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.llms import OpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import DeepLake
from langchain.text_splitter import CharacterTextSplitter

In [ ]:
# Load the pdf
loader = PyPDFLoader("Enqurious_ETL_DB document.pdf")

In [ ]:
# Now split the pdf into pages
pages = loader.load_and_split()

In [ ]:
len(pages)

In [ ]:
# Now split the documents via CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

In [ ]:
docs = text_splitter.split_documents(pages)

In [ ]:
len(docs)

In [ ]:
# Now create an embedding object
embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')

In [ ]:
# Now create a Deep lake object
from langchain.vectorstores import DeepLake

my_activeloop_org_id = "burhanuddinnahargarwala"
my_activeloop_datset_name = "enqurious_practice"
dataset_path = f"hub://{my_activeloop_org_id}/{my_activeloop_datset_name}"

[Dataset link](https://app.activeloop.ai/datasets/mydatasets/)

In [ ]:
db = DeepLake(dataset_path=dataset_path, embedding_function=embeddings)

In [ ]:
# Now add documents to the vector db
db.add_documents(docs)

In [ ]:
# Now fetch the retrievr from the db
retriever = db.as_retriever()

Testing part

In [ ]:
# Now use compressor to improve the retrieval process
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [ ]:
# create a llm object
llm = OpenAI(model_name="text-davinci-003", temperature=0, max_retries=3)

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

In [ ]:
response = compression_retriever.get_relevant_documents(
    "Give a brief summary of Enqurious ETL DB."
)

In [ ]:
for i in range(len(response)):
    print(response[i].page_content)

In [ ]:
response = compression_retriever.get_relevant_documents(
    "How many fact tables are there in ETL DB?"
)

In [ ]:
print(response[0].page_content)

In [ ]:
response = compression_retriever.get_relevant_documents(
    "Please provide the data dictionary for the table clients."
)

In [ ]:
print(response[0].page_content)

In [ ]:
response = compression_retriever.get_relevant_documents(
    "Which is the most granular attribute in skills_fact table?"
)

In [ ]:
print(response[1].page_content)

In [ ]:
from langchain.agents import initialize_agent
from langchain.tools import Tool

In [ ]:
tools = [
    Tool(
        name="Enqurious ETL Document",
        description="Contains the answers of all queries regarding enqurious extraction, transformation and loading, or any question reated to facts tables and it's attribute can be answered via this tool",
        func=compression_retriever.get_relevant_documents
    )
]

In [ ]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    verbose=True
)

In [ ]:
agent.run("How many tables are there in etl db data dictionary?")

In [ ]:
agent.run("Give the name of all the fact tables of etl db data dictionary?")

In [ ]:
agent.run("Give the name of all the fact tables of etl db data dictionary.")

In [ ]:
agent.run("List down all the tables of etl db data dictionary in bullet list format")

From the above output we can concluse that still we are not getting appropriate outcome

In [ ]:
# use retrieval chain
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

In [ ]:
tools = [
    Tool(
        name="Enqurious ETL Document",
        description="Contains the answers of all queries regarding enqurious extraction, transformation and loading, or any question reated to facts tables and it's attribute can be answered via this tool",
        func=qa_chain.run
    )
]

In [ ]:
from langchain.agents import AgentType

In [ ]:
agent2 = initialize_agent(
    tools=tools,
    llm=llm,
    verbose=True
)

In [ ]:
agent2.run("How can we perform merge operation between skills_fact and progress_fact table?")

In [ ]:
agent2.run(input="Give the name of all the fact tables of etl db data dictionary.")

In [ ]:
print('- clients \n- progress_fact \n- skills_fact \n- feedback_fact \n- learners_fact \n- order_duration_fact')

In [ ]:
agent2.run("List down all the tables of etl db data dictionary in bullet list format")

We can see both the agents are working in similar manner.

In [ ]:
agent2.run("Which attribute can be used to perform merge operation between skills_fact and progress_fact?")